# The Jena weather data, and a baseline nothing beats for a while

Eight years of readings every ten minutes, a windowed dataset, and a common-sense baseline that several neural networks will fail to beat.

**Runs on:** CPU — about 3 minutes &nbsp;·&nbsp; **Slides:** [Chapter 13 — Timeseries Forecasting](../../../course-web-slides/ch13/index.html) &nbsp;·&nbsp; **Section:** 01 — A temperature-forecasting example

---

## The data

In [ ]:
import os
import numpy as np
import keras

fname = keras.utils.get_file(
    origin="https://storage.googleapis.com/tensorflow/tf-keras-datasets/"
           "jena_climate_2009_2016.csv.zip",
    fname="jena_climate_2009_2016.csv.zip", extract=True)
csv_path = os.path.join(fname, "jena_climate_2009_2016.csv")

with open(csv_path) as f:
    data = f.read()

lines = data.split("\n")
header = lines[0].split(",")
lines = [l for l in lines[1:] if l]
print(len(header), "columns,", len(lines), "rows")
print(header)

In [ ]:
temperature = np.zeros((len(lines),))
raw_data = np.zeros((len(lines), len(header) - 1))
for i, line in enumerate(lines):
    values = [float(x) for x in line.split(",")[1:]]
    temperature[i] = values[1]
    raw_data[i, :] = values[:]

print(raw_data.shape)

## Look at it before modelling it

In [ ]:
import matplotlib.pyplot as plt

fig, (a1, a2) = plt.subplots(2, 1, figsize=(12, 6))
a1.plot(range(len(temperature)), temperature, lw=.4)
a1.set_title("Temperature, all eight years"); a1.set_ylabel("degC")
a2.plot(range(1440), temperature[:1440], lw=1)
a2.set_title("The first ten days"); a2.set_xlabel("10-minute steps")
plt.tight_layout(); plt.show()

Two periodicities, immediately visible: **yearly**, and **daily**. The yearly one is why a naive *predict the annual average* model would be poor, and the daily one is why *predict the last value* will be surprisingly good.

## The split, and why it is not random

In [ ]:
num_train_samples = int(0.5 * len(raw_data))
num_val_samples = int(0.25 * len(raw_data))
num_test_samples = len(raw_data) - num_train_samples - num_val_samples

print(f"train {num_train_samples}   validation {num_val_samples}   "
      f"test {num_test_samples}")

> ⚠️ **Chronological, always.** The validation and test data must be *posterior* to the training data. A random split lets the model interpolate between readings it has already seen, and the score becomes meaningless — chapter 5's notebook 03 drew the picture.

## Normalizing with training statistics

In [ ]:
mean = raw_data[:num_train_samples].mean(axis=0)
std = raw_data[:num_train_samples].std(axis=0)
raw_data -= mean
raw_data /= std
print("normalized using the first half only")

## Windowing

In [ ]:
sampling_rate = 6      # one reading per hour
sequence_length = 120  # five days of history
delay = sampling_rate * (sequence_length + 24 - 1)   # target is 24h ahead
batch_size = 256

train_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    shuffle=True,
    batch_size=batch_size,
    start_index=0,
    end_index=num_train_samples)

val_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay], targets=temperature[delay:],
    sampling_rate=sampling_rate, sequence_length=sequence_length,
    shuffle=True, batch_size=batch_size,
    start_index=num_train_samples,
    end_index=num_train_samples + num_val_samples)

test_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay], targets=temperature[delay:],
    sampling_rate=sampling_rate, sequence_length=sequence_length,
    shuffle=True, batch_size=batch_size,
    start_index=num_train_samples + num_val_samples)

for samples, targets in train_dataset:
    print("samples shape:", samples.shape)
    print("targets shape:", targets.shape)
    break

Expected output:

```
samples shape: (256, 120, 14)
targets shape: (256,)
```

**120 timesteps × 14 features per sample**, one temperature as the target. `sampling_rate=6` means we keep one reading per hour, so 120 steps is five days of history.

## The baseline

In [ ]:
def evaluate_naive_method(dataset):
    total_abs_err = 0.
    samples_seen = 0
    for samples, targets in dataset:
        # Column 1 is temperature; the last timestep is "now".
        preds = samples[:, -1, 1] * std[1] + mean[1]
        total_abs_err += np.sum(np.abs(preds - targets))
        samples_seen += samples.shape[0]
    return total_abs_err / samples_seen

print(f"validation MAE: {evaluate_naive_method(val_dataset):.2f} degC")
print(f"test MAE:       {evaluate_naive_method(test_dataset):.2f} degC")

Expected output:

```
validation MAE: 2.44 degC
test MAE:       2.62 degC
```

**Tomorrow will be like today.** Two and a half degrees of average error, from a model with no parameters.

That number is the bar. Chapter 6 said to compute a common-sense baseline before believing anything; here it takes four lines and several of the neural networks in the next notebook will not clear it.

## Understanding why the baseline is strong

In [ ]:
h = temperature[::6]                 # hourly
diffs = np.abs(h[24:] - h[:-24])    # change over 24 hours

plt.figure(figsize=(7, 4))
plt.hist(diffs, bins=80)
plt.axvline(diffs.mean(), color="r", ls="--",
            label=f"mean {diffs.mean():.2f} degC")
plt.xlabel("|temperature change over 24 hours|"); plt.legend()
plt.title("Why 'tomorrow is like today' is hard to beat")
plt.show()

The distribution is concentrated near zero. **Most of the time the temperature 24 hours from now is close to now**, and any model has to earn its keep on the tail.

---

## What to take away

- Timeseries splits are chronological. Never random.
- `timeseries_dataset_from_array` handles windowing, sampling rate, and the target offset.
- **The naive baseline is 2.44 °C** and several neural networks will not beat it.
- Understand *why* the baseline is strong before trying to beat it.